# GRU Training Results Viewer

This notebook is focused on three checks:

1. Loss convergence: training loss and validation loss across epochs.
2. Prediction quality and outlier inspection: measured vs predicted scatter plots in physical units.
3. Experiment comparison: show two experiments side by side using the same target signal.

The measured-vs-predicted scatter plot is useful for outlier checking: points far away from the diagonal line have large residuals and should be inspected.


In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import ipywidgets as widgets
from IPython.display import display, clear_output

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
EXPERIMENTS_DIR = PROJECT_ROOT / "results" / "kelmarsh" / "experiments"
print("Project root:", PROJECT_ROOT)
print("Experiments dir:", EXPERIMENTS_DIR)


Project root: D:\硕士论文\Project_vision1
Experiments dir: D:\硕士论文\Project_vision1\results\kelmarsh\experiments


In [2]:
def resolve_path(path_value):
    path = Path(path_value)
    return path if path.is_absolute() else PROJECT_ROOT / path


def discover_experiments():
    records = []
    for metadata_path in sorted(EXPERIMENTS_DIR.glob("*/metadata.json")):
        metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
        paths = {key: resolve_path(value) for key, value in metadata.get("paths", {}).items() if isinstance(value, str)}
        train_log_path = paths.get("train_log")
        prediction_path = paths.get("predictions")
        if train_log_path is None or prediction_path is None:
            continue
        if not train_log_path.exists() or not prediction_path.exists():
            continue

        label = (
            f"{metadata.get('run_id', metadata_path.parent.name)} | "
            f"best={metadata.get('best_epoch', '?')} | "
            f"val={metadata.get('best_val_loss', np.nan):.5g}"
        )
        records.append({
            "label": label,
            "run_id": metadata.get("run_id", metadata_path.parent.name),
            "created_at": metadata.get("created_at", ""),
            "metadata": metadata,
            "metadata_path": metadata_path,
            "train_log_path": train_log_path,
            "prediction_path": prediction_path,
        })

    records = sorted(records, key=lambda record: record["created_at"], reverse=True)
    if not records:
        raise FileNotFoundError(f"No completed experiments found in {EXPERIMENTS_DIR}")
    return records


experiments = discover_experiments()
print(f"Found {len(experiments)} experiment(s).")
for record in experiments:
    print("-", record["label"])


Found 1 experiment(s).
- all6_multi3_seq12_h64_l1_bs64_lr1e-03_wd0_do0_e60_p8_seed42 | best=45 | val=0.035991


In [3]:
def load_train_log(record):
    log = pd.read_csv(record["train_log_path"])
    required = {"epoch", "train_loss", "val_loss"}
    missing = required - set(log.columns)
    if missing:
        raise ValueError(f"Missing train log columns in {record['train_log_path']}: {missing}")
    return log


def load_predictions(record):
    pred = pd.read_csv(record["prediction_path"])
    if "Date and time" in pred.columns:
        pred["Date and time"] = pd.to_datetime(pred["Date and time"], errors="coerce")
    return pred


def prediction_targets(pred):
    prefix = "y_true::"
    return [column.removeprefix(prefix) for column in pred.columns if column.startswith(prefix)]


def target_columns(target):
    return f"y_true::{target}", f"y_pred::{target}", f"residual::{target}"


def experiment_summary(record):
    metadata = record["metadata"]
    keys = [
        "turbines",
        "epochs_requested",
        "epochs_completed",
        "best_epoch",
        "best_val_loss",
        "sequence_length",
        "batch_size",
        "hidden_size",
        "num_layers",
        "dropout",
        "learning_rate",
        "weight_decay",
        "amsgrad",
        "optimizer",
        "loss_function",
    ]
    rows = []
    for key in keys:
        if key in metadata:
            rows.append({"field": key, "value": metadata[key]})
    return pd.DataFrame(rows)


def residual_summary(pred, target):
    true_col, pred_col, residual_col = target_columns(target)
    df = pred[[true_col, pred_col] + ([residual_col] if residual_col in pred.columns else [])].copy()
    df = df.rename(columns={true_col: "measured", pred_col: "predicted", residual_col: "residual"})
    df = df.dropna(subset=["measured", "predicted"])
    if "residual" not in df.columns:
        df["residual"] = df["measured"] - df["predicted"]
    error = df["predicted"] - df["measured"]
    abs_residual = df["residual"].abs()
    return pd.DataFrame({
        "n": [len(df)],
        "MAE": [abs_residual.mean()],
        "RMSE": [np.sqrt(np.mean(np.square(error)))],
        "residual_p95_abs": [abs_residual.quantile(0.95)],
        "residual_p99_abs": [abs_residual.quantile(0.99)],
        "max_abs_residual": [abs_residual.max()],
    })


def sample_predictions(pred, target, max_points=30000, random_state=42):
    true_col, pred_col, residual_col = target_columns(target)
    cols = ["Date and time", "turbine_id", "segment_id", true_col, pred_col]
    cols = [column for column in cols if column in pred.columns]
    df = pred[cols].copy()
    df = df.rename(columns={true_col: "measured", pred_col: "predicted"})
    df = df.dropna(subset=["measured", "predicted"])
    df["residual"] = df["measured"] - df["predicted"]
    df["abs_residual"] = df["residual"].abs()
    if len(df) > max_points:
        return df.sample(max_points, random_state=random_state)
    return df


def top_outliers(pred, target, n=10):
    df = sample_predictions(pred, target, max_points=len(pred))
    cols = [column for column in ["Date and time", "turbine_id", "segment_id", "measured", "predicted", "residual", "abs_residual"] if column in df.columns]
    return df.sort_values("abs_residual", ascending=False)[cols].head(n)


In [4]:
def plot_loss_convergence(record, ax=None, title_suffix=""):
    log = load_train_log(record)
    if ax is None:
        _, ax = plt.subplots(figsize=(8, 4.5))
    ax.plot(log["epoch"], log["train_loss"], marker="o", markersize=3, linewidth=1.5, label="train loss")
    ax.plot(log["epoch"], log["val_loss"], marker="o", markersize=3, linewidth=1.5, label="validation loss")
    best_idx = log["val_loss"].idxmin()
    best_epoch = int(log.loc[best_idx, "epoch"])
    best_val = float(log.loc[best_idx, "val_loss"])
    ax.axvline(best_epoch, color="black", linestyle="--", linewidth=1, alpha=0.7)
    ax.scatter([best_epoch], [best_val], color="black", zorder=5, label=f"best epoch {best_epoch}")
    ax.set_title("Loss convergence" + title_suffix)
    ax.set_xlabel("Epoch")
    ax.set_ylabel("MSE loss (scaled target space)")
    ax.legend()
    return ax


def plot_measured_vs_predicted(record, target, ax=None, max_points=30000, outlier_n=30, title_suffix=""):
    pred = load_predictions(record)
    df = sample_predictions(pred, target, max_points=max_points)
    if ax is None:
        _, ax = plt.subplots(figsize=(6, 6))

    sns.scatterplot(
        data=df,
        x="measured",
        y="predicted",
        hue="abs_residual",
        palette="viridis",
        s=10,
        alpha=0.45,
        linewidth=0,
        ax=ax,
        legend=False,
    )

    lo = min(df["measured"].min(), df["predicted"].min())
    hi = max(df["measured"].max(), df["predicted"].max())
    ax.plot([lo, hi], [lo, hi], color="black", linestyle="--", linewidth=1, label="ideal: measured = predicted")

    outliers = df.nlargest(min(outlier_n, len(df)), "abs_residual")
    ax.scatter(
        outliers["measured"],
        outliers["predicted"],
        facecolors="none",
        edgecolors="red",
        s=45,
        linewidths=1.1,
        label=f"top {len(outliers)} residuals",
    )

    ax.set_title("Measured vs predicted" + title_suffix + chr(10) + target)
    ax.set_xlabel("Measured value")
    ax.set_ylabel("Predicted value")
    ax.legend(loc="best")
    ax.set_aspect("equal", adjustable="box")
    return ax


def render_single_experiment(record, target, max_points=30000, outlier_n=30):
    print(record["label"])
    display(experiment_summary(record))

    pred = load_predictions(record)
    if target not in prediction_targets(pred):
        raise ValueError(f"Target not found in prediction file: {target}")

    fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
    plot_loss_convergence(record, ax=axes[0])
    plot_measured_vs_predicted(record, target, ax=axes[1], max_points=max_points, outlier_n=outlier_n)
    fig.tight_layout()
    plt.show()

    print("Largest residual time points for the selected target:")
    display(top_outliers(pred, target, n=10))


def render_experiment_comparison(record_a, record_b, target, max_points=30000, outlier_n=30):
    fig, axes = plt.subplots(2, 2, figsize=(15, 11))
    suffix_a = chr(10) + record_a["run_id"]
    suffix_b = chr(10) + record_b["run_id"]
    plot_loss_convergence(record_a, ax=axes[0, 0], title_suffix=suffix_a)
    plot_loss_convergence(record_b, ax=axes[0, 1], title_suffix=suffix_b)
    plot_measured_vs_predicted(record_a, target, ax=axes[1, 0], max_points=max_points, outlier_n=outlier_n, title_suffix=suffix_a)
    plot_measured_vs_predicted(record_b, target, ax=axes[1, 1], max_points=max_points, outlier_n=outlier_n, title_suffix=suffix_b)
    fig.tight_layout()
    plt.show()

    rows = []
    for label, record in [("A", record_a), ("B", record_b)]:
        summary = residual_summary(load_predictions(record), target).assign(experiment=label, run_id=record["run_id"])
        rows.append(summary)
    display(pd.concat(rows, ignore_index=True)[["experiment", "run_id", "n", "MAE", "RMSE", "residual_p95_abs", "residual_p99_abs", "max_abs_residual"]])


In [5]:
first_pred = load_predictions(experiments[0])
target_options = prediction_targets(first_pred)

experiment_dropdown = widgets.Dropdown(
    options=[(record["label"], index) for index, record in enumerate(experiments)],
    value=0,
    description="Experiment:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="95%"),
)

target_dropdown = widgets.Dropdown(
    options=target_options,
    value=target_options[0] if target_options else None,
    description="Target:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="95%"),
)

max_points_slider = widgets.IntSlider(
    value=30000,
    min=5000,
    max=100000,
    step=5000,
    description="Scatter points:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="95%"),
)

outlier_slider = widgets.IntSlider(
    value=30,
    min=0,
    max=100,
    step=5,
    description="Marked outliers:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="95%"),
)

single_output = widgets.Output()


def update_single_view(change=None):
    with single_output:
        clear_output(wait=True)
        record = experiments[experiment_dropdown.value]
        render_single_experiment(
            record,
            target_dropdown.value,
            max_points=max_points_slider.value,
            outlier_n=outlier_slider.value,
        )

for widget in [experiment_dropdown, target_dropdown, max_points_slider, outlier_slider]:
    widget.observe(update_single_view, names="value")

display(widgets.VBox([experiment_dropdown, target_dropdown, max_points_slider, outlier_slider]), single_output)
update_single_view()


Output()

In [6]:
experiment_a_dropdown = widgets.Dropdown(
    options=[(record["label"], index) for index, record in enumerate(experiments)],
    value=0,
    description="Experiment A:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="95%"),
)

experiment_b_dropdown = widgets.Dropdown(
    options=[(record["label"], index) for index, record in enumerate(experiments)],
    value=1 if len(experiments) > 1 else 0,
    description="Experiment B:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="95%"),
)

comparison_target_dropdown = widgets.Dropdown(
    options=target_options,
    value=target_options[0] if target_options else None,
    description="Target:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="95%"),
)

comparison_output = widgets.Output()


def update_comparison(change=None):
    with comparison_output:
        clear_output(wait=True)
        if len(experiments) < 2:
            print("Only one experiment is available. Train another run to use the comparison view.")
            return
        render_experiment_comparison(
            experiments[experiment_a_dropdown.value],
            experiments[experiment_b_dropdown.value],
            comparison_target_dropdown.value,
            max_points=max_points_slider.value,
            outlier_n=outlier_slider.value,
        )

for widget in [experiment_a_dropdown, experiment_b_dropdown, comparison_target_dropdown, max_points_slider, outlier_slider]:
    widget.observe(update_comparison, names="value")

display(widgets.VBox([experiment_a_dropdown, experiment_b_dropdown, comparison_target_dropdown]), comparison_output)
update_comparison()


Output()